In [7]:
import os
import gc
import torch
import random
import time
from dotenv import load_dotenv
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)

load_dotenv()
hf_token = os.getenv("HF_TOKEN")
device = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint = "Helsinki-NLP/opus-mt-es-en"

def train_with_params(lr, ep, ls, trial_num):
    print(f"\n>>> INICIANDO TRIAL {trial_num}: LR={lr}, Epochs={ep}, Smoothing={ls}")
    
    # 1. Cargar y preparar datos (80% train, 20% val para rapidez)
    dataset = load_dataset('csv', data_files={'train': 'data/processed/train_cleaned.csv'})['train']
    dataset = dataset.train_test_split(test_size=0.2, seed=42)
    
    tokenizer = AutoTokenizer.from_pretrained(checkpoint, token=hf_token)

    def preprocess(examples):
        return tokenizer(
            [str(x) for x in examples["MSLG"]], 
            text_target=[str(x) for x in examples["SPA"]], 
            max_length=128, 
            truncation=True, 
            padding="max_length"
        )

    tokenized_train = dataset["train"].map(preprocess, batched=True)
    tokenized_val = dataset["test"].map(preprocess, batched=True)

    # 2. Configurar rutas únicas
    timestamp = int(time.time())
    output_path = f"./models/random_search/trial_{trial_num}_LR{lr}_EP{ep}_LS{ls}_{timestamp}"

    # 3. Modelo
    model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint, token=hf_token).to(device)

    # 4. Argumentos de entrenamiento
    args = Seq2SeqTrainingArguments(
        output_dir=output_path,
        eval_strategy="epoch",
        learning_rate=lr,
        num_train_epochs=ep,
        label_smoothing_factor=ls,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        weight_decay=0.01,
        predict_with_generate=True,
        fp16=True,
        save_total_limit=1, # Solo guarda el mejor para no llenar el disco
        logging_steps=10,
        report_to="none"
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        processing_class=tokenizer, 
        data_collator=DataCollatorForSeq2Seq(tokenizer, model=model)
    )

    trainer.train()
    trainer.save_model(f"{output_path}/final_model")
    
    # 5. LIMPIEZA CRÍTICA DE MEMORIA
    del model
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
    print(f"--- Trial {trial_num} finalizado y guardado en {output_path} ---\n")

if __name__ == "__main__":
    # Definir espacio de búsqueda
    learning_rates = [2e-5, 3e-5, 5e-5]
    epochs_options = [20, 35, 50] # Más épocas para combatir el inglés
    smoothing_options = [0.1, 0.15, 0.2]

    num_trials = 3 # Puedes subirlo a 5 si tienes tiempo
    
    print(f"Iniciando Random Search en {device}...")
    
    for i in range(num_trials):
        selected_lr = random.choice(learning_rates)
        selected_ep = random.choice(epochs_options)
        selected_ls = random.choice(smoothing_options)
        
        try:
            train_with_params(selected_lr, selected_ep, selected_ls, i+1)
        except Exception as e:
            print(f"Error en trial {i+1}: {e}")
            torch.cuda.empty_cache()
            continue

ModuleNotFoundError: No module named 'torch'